# 텍스트 임베딩

## CTA로 파인튜닝된 모델에 텍스트 임베딩 예측 수행

id2label로 숫자 → 라벨 명칭 복원

predict_label()은 단일 문장을 넣으면 (라벨명, 확률) 반환

model.eval() 상태에서 추론

BERT 토크나이저는 max_length=32 기준 유지



In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import json

# 1. label_map 불러오기
with open("label_map.json", "r", encoding="utf-8") as f:
    label_map = json.load(f)
id2label = {v: k for k, v in label_map.items()}
# 2. KoBERT tokenizer & 모델 불러오기
tokenizer = BertTokenizer.from_pretrained("monologg/kobert")
bertmodel = BertModel.from_pretrained("monologg/kobert")

# 3. 분류기 정의
class NewClassifier(nn.Module):
    def __init__(self, bert, hidden_size=768, num_classes=len(label_map)):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        return self.classifier(pooled)

# 4. 모델 로딩
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NewClassifier(bertmodel).to(device)
model.load_state_dict(torch.load("new_kobert_sti_finetuned.pt", map_location=device))
model.eval()

# 5. CSV 데이터 불러오기
df = pd.read_csv("inference_input.csv", encoding="utf-8")

# 6. 예측 함수
def predict(text):
    encoded = tokenizer(text, padding='max_length', truncation=True, max_length=32, return_tensors="pt")
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        pred_id = torch.argmax(logits, dim=1).item()
        prob = torch.softmax(logits, dim=1)[0][pred_id].item()

    return id2label[pred_id], prob

# 7. 예측 수행
print("CSV 파일 로드 완료: inference_input.csv")
print("예측 모델 로딩 완료: new_kobert_sti_finetuned.pt")
print(f"총 {len(df)}건 데이터 → 예측 수행 중...\n")

df['pred_label'] = df['text'].apply(lambda x: predict(x)[0])
df['confidence'] = df['text'].apply(lambda x: predict(x)[1])

# 8. 출력용 결과 정렬 및 자르기
sorted_df = df.sort_values(by='confidence', ascending=False).reset_index(drop=True)
print("===== 예측 결과 샘플 =====")
for i, row in sorted_df.head(100).iterrows():
    print(f"텍스트: {row['text'][:100]}")
    print(f" → 예측 라벨: {row['pred_label']} | 신뢰도: {row['confidence']:.4f}\n")

# 9. 요약 통계 출력
total = len(df)
count = (df['pred_label'] == '사건개요').sum()
avg_conf = df[df['pred_label'] == '사건개요']['confidence'].mean()
low_conf = df[df['confidence'] < 0.2]['confidence'].mean()

print("===== 요약 통계 =====")
print(f"전체 사건개요로 분류된 행: {count}/{total}")
print(f"사건개요 평균 confidence: {avg_conf:.4f}")
print(f"confidence 0.2 미만 평균: {low_conf:.4f}\n")

# 10. 저장
df.to_csv("inference_output.csv", index=False, encoding="utf-8-sig")
print("결과 저장 완료: inference_output.csv")
